# Prepare paired spectral datasets

English documentation and portable paths have been added to the original research workflow. Numerical calculations, model architecture, losses, and experimental settings are preserved. Outputs are cleared. Read [reproducibility notes](../docs/REPRODUCIBILITY.md) before execution.


In [ ]:
from pathlib import Path
import os
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "scripts" / "project_paths.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start Jupyter from the repository root or its notebooks directory.")
sys.path.insert(0, str(ROOT / "scripts"))
from project_paths import create_run
DATA_DIR, OUTPUT_DIR = create_run('preprocessing')


## Construct augmented training pairs

Original code cell 1.


In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import itertools
import h5py


df = pd.read_csv(str(DATA_DIR / 'measurements/20condition_seq1_15/mean.csv'))
df = df[df['Cr'].isin([1, 2, 3, 4, 5, 7, 9, 10, 11, 13, 15])]
col = list(df.columns)
numeric_cols = col[11:11+1873]
wavelength = np.array([float(j) for j in numeric_cols])


ranges = [
    {"x0": 405.24, "x1": 407.245},
    {"x0": 367.942, "x1": 369.659}
]
col_map = {round(float(c), 3): c for c in numeric_cols}
for r in ranges:
    x0, x1 = r["x0"], r["x1"]
    lower_val = min(wavelength, key=lambda w: abs(w - x0))
    upper_val = min(wavelength, key=lambda w: abs(w - x1))
    lower_col = col_map[round(lower_val, 3)]
    upper_col = col_map[round(upper_val, 3)]
    for w in wavelength:
        if x0 <= w <= x1:
            col_str = col_map.get(round(w, 3))
            if col_str is None or 'Pb' in col_str:
                continue
            y0 = df[lower_col]
            y1 = df[upper_col]
            a = (y1 - y0) / (upper_val - lower_val)
            b = y0 - a * lower_val
            df[col_str] = a * w + b


input_cols = sorted(numeric_cols, key=lambda x: float(x))
target_cols_sorted = sorted(numeric_cols, key=lambda x: float(x))

def get_index_range(w1, w2):
    return np.array([input_cols.index(f"{w1:.3f}"), input_cols.index(f"{w2:.3f}")])

Zn_index = get_index_range(212.129, 216.777)
Ni_index = get_index_range(229.563, 234.734)
Cu_index = get_index_range(326.669, 330.016)
metal_peaks = {"Cu": [Cu_index], "Ni": [Ni_index], "Zn": [Zn_index]}

zero_index = np.concatenate([
    np.arange(*Zn_index + 1),
    np.arange(*Ni_index + 1),
    np.arange(*Cu_index + 1),
])


ppm_values = [0, 1, 2, 3, 4, 5]
combinations = list(itertools.product(ppm_values, repeat=3))

cr1_group = df[df["Cr"] == 1]
global_y_background = cr1_group[target_cols_sorted].mean().copy()
global_y_background.iloc[zero_index] = 0

metal_peak_cache_y_cr1 = {}
for metal in metal_peaks:
    metal_peak_cache_y_cr1[metal] = {}
    for ppm in ppm_values:
        row = cr1_group[cr1_group[metal] == ppm]
        if not row.empty:
            metal_peak_cache_y_cr1[metal][ppm] = row.iloc[0][target_cols_sorted].values
        else:
            metal_peak_cache_y_cr1[metal][ppm] = None


X_data = []
Y_data = []

for cr_value in tqdm(df['Cr'].unique(), desc="Augmenting per Cr group"):
    group = df[df["Cr"] == cr_value]
    background_x = group[input_cols].mean().copy()
    background_x.iloc[zero_index] = 0

    metal_peak_cache_x = {}
    for metal in metal_peaks:
        metal_peak_cache_x[metal] = {}
        for ppm in ppm_values:
            row = group[group[metal] == ppm]
            if not row.empty:
                metal_peak_cache_x[metal][ppm] = row.iloc[0][input_cols].values
            else:
                print(f"⚠️ Warning: Cr=1, {metal} = {ppm} is missing.")
                metal_peak_cache_y_cr1[metal][ppm] = None

    for cu, ni, zn in combinations:
        combo = {"Cu": cu, "Ni": ni, "Zn": zn}
        spectrum_x = background_x.copy()
        for metal in combo:
            sx = metal_peak_cache_x[metal][combo[metal]]
            if sx is not None:
                for indices in metal_peaks[metal]:
                    spectrum_x.iloc[indices[0]:indices[1]+1] = pd.Series(
                        sx[indices[0]:indices[1]+1],
                        index=input_cols[indices[0]:indices[1]+1]
                    )

        spectrum_y = global_y_background.copy()
        for metal in combo:
            sy = metal_peak_cache_y_cr1[metal][combo[metal]]
            if sy is not None:
                for indices in metal_peaks[metal]:
                    spectrum_y.iloc[indices[0]:indices[1]+1] = pd.Series(
                        sy[indices[0]:indices[1]+1],
                        index=input_cols[indices[0]:indices[1]+1]
                    )

        X_data.append([cr_value, cu, ni, zn] + list(spectrum_x.values))
        Y_data.append([cr_value, cu, ni, zn] + list(spectrum_y.values))


X_array = np.array(X_data)
Y_array = np.array(Y_data)


X_spectra = X_array[:, 4:]
Y_spectra = Y_array[:, 4:]

df1 = pd.read_csv(str(DATA_DIR / 'measurements/20condition_seq1_15/mean.csv'))
df2 = pd.read_csv(str(DATA_DIR / 'measurements/20condition_seq16_19/mean.csv'))


df1_spectra = df1[input_cols].values
df2_spectra = df2[input_cols].values

over_limit_df1 = np.any(df1_spectra > 60000, axis=0)
over_limit_df2 = np.any(df2_spectra > 60000, axis=0)
over_limit_total = over_limit_df1 | over_limit_df2


mask_low_wl = wavelength < 350
low_indices = np.where(mask_low_wl)[0]
indices_set_to_60000_low = low_indices[over_limit_total[low_indices]]


X_spectra[:, indices_set_to_60000_low] = 60000
Y_spectra[:, indices_set_to_60000_low] = 60000


mask_high_wl = wavelength >= 350
high_indices = np.where(mask_high_wl)[0]
X_spectra[:, high_indices] = np.clip(X_spectra[:, high_indices], None, 60000)
Y_spectra[:, high_indices] = np.clip(Y_spectra[:, high_indices], None, 60000)


mask_condition3 = (wavelength >= 323) & (wavelength <= 325)
indices_condition3 = np.where(mask_condition3)[0]

X_spectra[:, indices_condition3] = 60000
Y_spectra[:, indices_condition3] = 60000


X_array[:, 4:] = X_spectra
Y_array[:, 4:] = Y_spectra


X_cols = ["Cr", "Cu", "Ni", "Zn"] + input_cols
Y_cols = ["Cr", "Cu", "Ni", "Zn"] + target_cols_sorted

with h5py.File(str(OUTPUT_DIR / 'processed/ml_20condition_training_data.h5'), "w") as h5f:
    h5f.create_dataset("X_with_meta", data=X_array)
    h5f.create_dataset("X_columns", data=np.array(X_cols, dtype='S'))

with h5py.File(str(OUTPUT_DIR / 'processed/ml_20condition_training_label.h5'), "w") as h5f:
    h5f.create_dataset("Y_with_meta", data=Y_array)
    h5f.create_dataset("Y_columns", data=np.array(Y_cols, dtype='S'))


X_df = pd.DataFrame(X_array, columns=X_cols)
Y_df = pd.DataFrame(Y_array, columns=Y_cols)

X_df.to_csv(str(OUTPUT_DIR / 'processed/ml_20condition_training_data.csv'), index=False)
Y_df.to_csv(str(OUTPUT_DIR / 'processed/ml_20condition_training_label.csv'), index=False)

print("Saved the training dataset.")
print(f"Generated {len(X_df)} samples")


## Construct test dataset 1

Original code cell 2.


In [ ]:
df = pd.read_csv(str(DATA_DIR / 'measurements/20condition_seq1_15/mean.csv'))
df = df[df['Cr'].isin([2, 4, 6, 8, 12, 14])]
col = list(df.columns)
numeric_cols = col[11:11+1873]
wavelength = np.array([float(j) for j in numeric_cols])


ranges = [
    {"x0": 405.24, "x1": 407.245},
    {"x0": 367.942, "x1": 369.659}
]
col_map = {round(float(c), 3): c for c in numeric_cols}
for r in ranges:
    x0, x1 = r["x0"], r["x1"]
    lower_val = min(wavelength, key=lambda w: abs(w - x0))
    upper_val = min(wavelength, key=lambda w: abs(w - x1))
    lower_col = col_map[round(lower_val, 3)]
    upper_col = col_map[round(upper_val, 3)]
    for w in wavelength:
        if x0 <= w <= x1:
            col_str = col_map.get(round(w, 3))
            if col_str is None or 'Pb' in col_str:
                continue
            y0 = df[lower_col]
            y1 = df[upper_col]
            a = (y1 - y0) / (upper_val - lower_val)
            b = y0 - a * lower_val
            df[col_str] = a * w + b


input_cols = sorted(numeric_cols, key=lambda x: float(x))
target_cols_sorted = sorted(numeric_cols, key=lambda x: float(x))

def get_index_range(w1, w2):
    return np.array([input_cols.index(f"{w1:.3f}"), input_cols.index(f"{w2:.3f}")])

Zn_index = get_index_range(212.129, 216.777)
Ni_index = get_index_range(229.563, 234.734)
Cu_index = get_index_range(326.669, 330.016)
metal_peaks = {"Cu": [Cu_index], "Ni": [Ni_index], "Zn": [Zn_index]}

zero_index = np.concatenate([
    np.arange(*Zn_index + 1),
    np.arange(*Ni_index + 1),
    np.arange(*Cu_index + 1),
])


ppm_values = [0, 1, 2, 3, 4, 5]
combinations = list(itertools.product(ppm_values, repeat=3))


original_df = pd.read_csv(str(DATA_DIR / 'measurements/20condition_seq1_15/mean.csv'))
cr1_group = original_df[original_df["Cr"] == 1]
global_y_background = cr1_group[target_cols_sorted].mean().copy()
global_y_background.iloc[zero_index] = 0

metal_peak_cache_y_cr1 = {}
for metal in metal_peaks:
    metal_peak_cache_y_cr1[metal] = {}
    for ppm in ppm_values:
        row = cr1_group[cr1_group[metal] == ppm]
        if not row.empty:
            metal_peak_cache_y_cr1[metal][ppm] = row.iloc[0][target_cols_sorted].values
        else:
            metal_peak_cache_y_cr1[metal][ppm] = None


X_data = []
Y_data = []

for _, row in df.iterrows():
    cr, cu, ni, zn = row["Cr"], row["Cu"], row["Ni"], row["Zn"]
    spectrum_x = row[input_cols].values
    X_data.append([cr, cu, ni, zn] + list(spectrum_x))


    spectrum_y = global_y_background.copy()
    for metal in ["Cu", "Ni", "Zn"]:
        sy = metal_peak_cache_y_cr1[metal].get(locals()[metal.lower()])
        if sy is not None:
            for indices in metal_peaks[metal]:
                spectrum_y[indices[0]:indices[1]+1] = sy[indices[0]:indices[1]+1]

    Y_data.append([cr, cu, ni, zn] + list(spectrum_y))


X_array = np.array(X_data)
Y_array = np.array(Y_data)


X_spectra = X_array[:, 4:]
Y_spectra = Y_array[:, 4:]


X_spectra[:, indices_set_to_60000_low] = 60000
Y_spectra[:, indices_set_to_60000_low] = 60000


mask_high_wl = wavelength >= 350
high_indices = np.where(mask_high_wl)[0]
X_spectra[:, high_indices] = np.clip(X_spectra[:, high_indices], None, 60000)
Y_spectra[:, high_indices] = np.clip(Y_spectra[:, high_indices], None, 60000)


mask_condition3 = (wavelength >= 323) & (wavelength <= 325)
indices_condition3 = np.where(mask_condition3)[0]

X_spectra[:, indices_condition3] = 60000
Y_spectra[:, indices_condition3] = 60000


X_array[:, 4:] = X_spectra
Y_array[:, 4:] = Y_spectra


X_cols = ["Cr", "Cu", "Ni", "Zn"] + input_cols
Y_cols = ["Cr", "Cu", "Ni", "Zn"] + target_cols_sorted

with h5py.File(str(OUTPUT_DIR / 'processed/ml_20condition_testing_data1.h5'), "w") as h5f:
    h5f.create_dataset("X_with_meta", data=X_array)
    h5f.create_dataset("X_columns", data=np.array(X_cols, dtype='S'))

with h5py.File(str(OUTPUT_DIR / 'processed/ml_20condition_testing_label1.h5'), "w") as h5f:
    h5f.create_dataset("Y_with_meta", data=Y_array)
    h5f.create_dataset("Y_columns", data=np.array(Y_cols, dtype='S'))


X_df = pd.DataFrame(X_array, columns=X_cols)
Y_df = pd.DataFrame(Y_array, columns=Y_cols)

X_df.to_csv(str(OUTPUT_DIR / 'processed/ml_20condition_testing_data1.csv'), index=False)
Y_df.to_csv(str(OUTPUT_DIR / 'processed/ml_20condition_testing_label1.csv'), index=False)

print("Saved test dataset 1.")
print(f"Generated {len(X_df)} samples")


## Construct test dataset 2

Original code cell 3.


In [ ]:
df = pd.read_csv(str(DATA_DIR / 'measurements/20condition_seq16_19/mean.csv'))
col = list(df.columns)
numeric_cols = col[11:11+1873]
wavelength = np.array([float(j) for j in numeric_cols])


ranges = [
    {"x0": 405.24, "x1": 407.245},
    {"x0": 367.942, "x1": 369.659}
]
col_map = {round(float(c), 3): c for c in numeric_cols}
for r in ranges:
    x0, x1 = r["x0"], r["x1"]
    lower_val = min(wavelength, key=lambda w: abs(w - x0))
    upper_val = min(wavelength, key=lambda w: abs(w - x1))
    lower_col = col_map[round(lower_val, 3)]
    upper_col = col_map[round(upper_val, 3)]
    for w in wavelength:
        if x0 <= w <= x1:
            col_str = col_map.get(round(w, 3))
            if col_str is None or 'Pb' in col_str:
                continue
            y0 = df[lower_col]
            y1 = df[upper_col]
            a = (y1 - y0) / (upper_val - lower_val)
            b = y0 - a * lower_val
            df[col_str] = a * w + b


input_cols = sorted(numeric_cols, key=lambda x: float(x))
target_cols_sorted = sorted(numeric_cols, key=lambda x: float(x))


label_df = pd.read_csv(str(DATA_DIR / 'measurements/20condition_seq1_15/mean.csv'))
label_df = label_df[label_df['Cr'] == 1]


X_data = []
Y_data = []

for _, row in df.iterrows():
    cr, cu, ni, zn = row["Cr"], row["Cu"], row["Ni"], row["Zn"]
    spectrum_x = row[input_cols].values
    X_data.append([cr, cu, ni, zn] + list(spectrum_x))


    spectrum_y = global_y_background.copy()
    for metal in ["Cu", "Ni", "Zn"]:
        sy = metal_peak_cache_y_cr1[metal].get(locals()[metal.lower()])
        if sy is not None:
            for indices in metal_peaks[metal]:
                spectrum_y[indices[0]:indices[1]+1] = sy[indices[0]:indices[1]+1]

    Y_data.append([cr, cu, ni, zn] + list(spectrum_y))


X_array = np.array(X_data)
Y_array = np.array(Y_data)


X_spectra = X_array[:, 4:]
Y_spectra = Y_array[:, 4:]


X_spectra[:, indices_set_to_60000_low] = 60000
Y_spectra[:, indices_set_to_60000_low] = 60000


mask_high_wl = wavelength >= 350
high_indices = np.where(mask_high_wl)[0]
X_spectra[:, high_indices] = np.clip(X_spectra[:, high_indices], None, 60000)
Y_spectra[:, high_indices] = np.clip(Y_spectra[:, high_indices], None, 60000)


mask_condition3 = (wavelength >= 323) & (wavelength <= 325)
indices_condition3 = np.where(mask_condition3)[0]

X_spectra[:, indices_condition3] = 60000
Y_spectra[:, indices_condition3] = 60000


X_array[:, 4:] = X_spectra
Y_array[:, 4:] = Y_spectra


with h5py.File(str(OUTPUT_DIR / 'processed/ml_20condition_testing_data2.h5'), "w") as h5f:
    h5f.create_dataset("X_with_meta", data=X_array)
    h5f.create_dataset("X_columns", data=np.array(X_cols, dtype='S'))

with h5py.File(str(OUTPUT_DIR / 'processed/ml_20condition_testing_label2.h5'), "w") as h5f:
    h5f.create_dataset("Y_with_meta", data=Y_array)
    h5f.create_dataset("Y_columns", data=np.array(Y_cols, dtype='S'))


X_cols = ["Cr", "Cu", "Ni", "Zn"] + input_cols
Y_cols = ["Cr", "Cu", "Ni", "Zn"] + target_cols_sorted

X_df = pd.DataFrame(X_array, columns=X_cols)
Y_df = pd.DataFrame(Y_array, columns=Y_cols)

X_df.to_csv(str(OUTPUT_DIR / 'processed/ml_20condition_testing_data2.csv'), index=False)
Y_df.to_csv(str(OUTPUT_DIR / 'processed/ml_20condition_testing_label2.csv'), index=False)

print("Paired and saved test dataset 2.")
print(f"Generated {len(X_df)} samples")


## Construct augmented repeat-measurement pairs

Original code cell 4.


In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import itertools
import h5py
import os


def normalize_columns(df):
    new_cols = []
    for c in df.columns:
        try:
            f = float(c)
            new_cols.append(f"{f:.3f}")
        except:
            new_cols.append(c)
    df.columns = new_cols
    return df


df = pd.read_csv(str(DATA_DIR / 'measurements/retake3/mean.csv'))
df = normalize_columns(df)


label_df = pd.read_csv(str(DATA_DIR / 'measurements/20condition_seq1_15/mean.csv'))
label_df = normalize_columns(label_df)


label_cr1 = label_df[label_df["Cr"] == 1].copy()


col = list(df.columns)
numeric_cols = col[11:11+1873]
wavelength = np.array([float(j) for j in numeric_cols])


target_cols_sorted = sorted(numeric_cols, key=lambda x: float(x))


global_y_background = label_cr1[target_cols_sorted].mean().copy()


ranges = [
    {"x0": 405.24, "x1": 407.245},
    {"x0": 367.942, "x1": 369.659}
]
col_map = {round(float(c), 3): c for c in numeric_cols}
for r in ranges:
    x0, x1 = r["x0"], r["x1"]
    lower_val = min(wavelength, key=lambda w: abs(w - x0))
    upper_val = min(wavelength, key=lambda w: abs(w - x1))
    lower_col = col_map[round(lower_val, 3)]
    upper_col = col_map[round(upper_val, 3)]
    for w in wavelength:
        if x0 <= w <= x1:
            col_str = col_map.get(round(w, 3))
            if col_str is None or 'Pb' in col_str:
                continue
            y0 = df[lower_col]
            y1 = df[upper_col]
            a = (y1 - y0) / (upper_val - lower_val)
            b = y0 - a * lower_val
            df[col_str] = a * w + b


input_cols = sorted(numeric_cols, key=lambda x: float(x))
target_cols_sorted = sorted(numeric_cols, key=lambda x: float(x))

def get_index_range(w1, w2):
    return np.array([input_cols.index(f"{w1:.3f}"), input_cols.index(f"{w2:.3f}")])

Zn_index = get_index_range(212.129, 216.777)
Ni_index = get_index_range(229.563, 234.734)
Cu_index = get_index_range(326.669, 330.016)
metal_peaks = {"Cu": [Cu_index], "Ni": [Ni_index], "Zn": [Zn_index]}


def inclusive_range(idx_pair):
    s, e = int(idx_pair[0]), int(idx_pair[1])
    return np.arange(s, e + 1)

zero_index = np.concatenate([
    inclusive_range(Zn_index),
    inclusive_range(Ni_index),
    inclusive_range(Cu_index),
])


label_df = pd.read_csv(str(DATA_DIR / 'measurements/20condition_seq1_15/mean.csv'))
label_cr1 = label_df[label_df["Cr"] == 1].copy()


global_y_background = label_cr1[target_cols_sorted].mean().copy()
global_y_background.iloc[zero_index] = 0


ppm_values = [0, 1, 2, 3, 4, 5]
metal_peak_cache_y_cr1 = {}
for metal in metal_peaks:
    metal_peak_cache_y_cr1[metal] = {}
    for ppm in ppm_values:
        row = label_cr1[label_cr1[metal] == ppm]
        if not row.empty:
            metal_peak_cache_y_cr1[metal][ppm] = row.iloc[0][target_cols_sorted].values
        else:
            metal_peak_cache_y_cr1[metal][ppm] = None

combinations = list(itertools.product(ppm_values, repeat=3))

# === 5. Augmentation ===
X_data = []
Y_data = []

for cr_value in tqdm(df['Cr'].unique(), desc="Augmenting per Cr group"):
    group = df[df["Cr"] == cr_value]

    background_x = group[input_cols].mean().copy()
    background_x.iloc[zero_index] = 0


    metal_peak_cache_x = {}
    for metal in metal_peaks:
        metal_peak_cache_x[metal] = {}
        for ppm in ppm_values:
            row = group[group[metal] == ppm]
            if not row.empty:
                metal_peak_cache_x[metal][ppm] = row.iloc[0][input_cols].values
            else:
                print(f"⚠️ Warning: Cr={cr_value}, {metal} = {ppm} is missing.")
                metal_peak_cache_x[metal][ppm] = None


    for cu, ni, zn in combinations:
        combo = {"Cu": cu, "Ni": ni, "Zn": zn}


        spectrum_x = background_x.copy()
        for metal in combo:
            sx = metal_peak_cache_x[metal][combo[metal]]
            if sx is not None:
                for indices in metal_peaks[metal]:
                    s, e = int(indices[0]), int(indices[1])
                    spectrum_x.iloc[s:e+1] = pd.Series(sx[s:e+1], index=input_cols[s:e+1])


        spectrum_y = global_y_background.copy()
        for metal in combo:
            sy = metal_peak_cache_y_cr1[metal][combo[metal]]
            if sy is not None:
                for indices in metal_peaks[metal]:
                    s, e = int(indices[0]), int(indices[1])
                    spectrum_y.iloc[s:e+1] = pd.Series(sy[s:e+1], index=input_cols[s:e+1])


        X_data.append([cr_value, cu, ni, zn] + list(spectrum_x.values))
        Y_data.append([1,        cu, ni, zn] + list(spectrum_y.values))


X_array = np.array(X_data)
Y_array = np.array(Y_data)

X_spectra = X_array[:, 4:]
Y_spectra = Y_array[:, 4:]


df1 = pd.read_csv(str(DATA_DIR / 'measurements/20condition_seq1_15/mean.csv'))
df2 = pd.read_csv(str(DATA_DIR / 'measurements/20condition_seq16_19/mean.csv'))

over_limit_total = (np.any(df1[input_cols].values > 60000, axis=0) |
                    np.any(df2[input_cols].values > 60000, axis=0))


low_indices = np.where(wavelength < 350)[0]
indices_set_to_60000_low = low_indices[over_limit_total[low_indices]]
X_spectra[:, indices_set_to_60000_low] = 60000
Y_spectra[:, indices_set_to_60000_low] = 60000


high_indices = np.where(wavelength >= 350)[0]
X_spectra[:, high_indices] = np.clip(X_spectra[:, high_indices], None, 60000)
Y_spectra[:, high_indices] = np.clip(Y_spectra[:, high_indices], None, 60000)


indices_condition3 = np.where((wavelength >= 323) & (wavelength <= 325))[0]
X_spectra[:, indices_condition3] = 60000
Y_spectra[:, indices_condition3] = 60000


X_array[:, 4:] = X_spectra
Y_array[:, 4:] = Y_spectra


os.makedirs("ML dataset", exist_ok=True)

X_cols = ["Cr", "Cu", "Ni", "Zn"] + input_cols
Y_cols = ["Cr", "Cu", "Ni", "Zn"] + target_cols_sorted

with h5py.File(str(OUTPUT_DIR / 'processed/ml_20condition_retake2_with_aug_data.h5'), "w") as h5f:
    h5f.create_dataset("X_with_meta", data=X_array)
    h5f.create_dataset("X_columns", data=np.array(X_cols, dtype='S'))

with h5py.File(str(OUTPUT_DIR / 'processed/ml_20condition_retake2_with_aug_label.h5'), "w") as h5f:
    h5f.create_dataset("Y_with_meta", data=Y_array)
    h5f.create_dataset("Y_columns", data=np.array(Y_cols, dtype='S'))

pd.DataFrame(X_array, columns=X_cols).to_csv(str(OUTPUT_DIR / 'processed/ml_20condition_retake3_with_aug_data.csv'), index=False)
pd.DataFrame(Y_array, columns=Y_cols).to_csv(str(OUTPUT_DIR / 'processed/ml_20condition_retake3_with_aug_label.csv'), index=False)

print("Saved the augmented repeat-measurement dataset.")
print(f"Generated {len(X_array)} samples; target Cr metadata uses condition 1 from sequences 1-15.")


## Construct repeat-measurement pairs

Original code cell 5.


In [ ]:
df = pd.read_csv(str(DATA_DIR / 'measurements/retake3/mean.csv'))
col = list(df.columns)
numeric_cols = col[11:11+1873]
wavelength = np.array([float(j) for j in numeric_cols])


ranges = [
    {"x0": 405.24, "x1": 407.245},
    {"x0": 367.942, "x1": 369.659}
]
col_map = {round(float(c), 3): c for c in numeric_cols}
for r in ranges:
    x0, x1 = r["x0"], r["x1"]
    lower_val = min(wavelength, key=lambda w: abs(w - x0))
    upper_val = min(wavelength, key=lambda w: abs(w - x1))
    lower_col = col_map[round(lower_val, 3)]
    upper_col = col_map[round(upper_val, 3)]
    for w in wavelength:
        if x0 <= w <= x1:
            col_str = col_map.get(round(w, 3))
            if col_str is None or 'Pb' in col_str:
                continue
            y0 = df[lower_col]
            y1 = df[upper_col]
            a = (y1 - y0) / (upper_val - lower_val)
            b = y0 - a * lower_val
            df[col_str] = a * w + b


input_cols = sorted(numeric_cols, key=lambda x: float(x))
target_cols_sorted = sorted(numeric_cols, key=lambda x: float(x))


label_df = pd.read_csv(str(DATA_DIR / 'measurements/retake3/mean.csv'))
label_df = label_df[label_df['Cr'] == 1]


X_data = []
Y_data = []

for _, row in df.iterrows():
    cr, cu, ni, zn = row["Cr"], row["Cu"], row["Ni"], row["Zn"]
    spectrum_x = row[input_cols].values
    X_data.append([cr, cu, ni, zn] + list(spectrum_x))


    spectrum_y = global_y_background.copy()
    for metal in ["Cu", "Ni", "Zn"]:
        sy = metal_peak_cache_y_cr1[metal].get(locals()[metal.lower()])
        if sy is not None:
            for indices in metal_peaks[metal]:
                spectrum_y[indices[0]:indices[1]+1] = sy[indices[0]:indices[1]+1]

    Y_data.append([cr, cu, ni, zn] + list(spectrum_y))


X_array = np.array(X_data)
Y_array = np.array(Y_data)


X_spectra = X_array[:, 4:]
Y_spectra = Y_array[:, 4:]


X_spectra[:, indices_set_to_60000_low] = 60000
Y_spectra[:, indices_set_to_60000_low] = 60000


mask_high_wl = wavelength >= 350
high_indices = np.where(mask_high_wl)[0]
X_spectra[:, high_indices] = np.clip(X_spectra[:, high_indices], None, 60000)
Y_spectra[:, high_indices] = np.clip(Y_spectra[:, high_indices], None, 60000)


mask_condition3 = (wavelength >= 323) & (wavelength <= 325)
indices_condition3 = np.where(mask_condition3)[0]

X_spectra[:, indices_condition3] = 60000
Y_spectra[:, indices_condition3] = 60000


X_array[:, 4:] = X_spectra
Y_array[:, 4:] = Y_spectra


with h5py.File(str(OUTPUT_DIR / 'processed/ml_20condition_retake3_data.h5'), "w") as h5f:
    h5f.create_dataset("X_with_meta", data=X_array)
    h5f.create_dataset("X_columns", data=np.array(X_cols, dtype='S'))

with h5py.File(str(OUTPUT_DIR / 'processed/ml_20condition_retake3_label.h5'), "w") as h5f:
    h5f.create_dataset("Y_with_meta", data=Y_array)
    h5f.create_dataset("Y_columns", data=np.array(Y_cols, dtype='S'))


X_cols = ["Cr", "Cu", "Ni", "Zn"] + input_cols
Y_cols = ["Cr", "Cu", "Ni", "Zn"] + target_cols_sorted

X_df = pd.DataFrame(X_array, columns=X_cols)
Y_df = pd.DataFrame(Y_array, columns=Y_cols)

X_df.to_csv(str(OUTPUT_DIR / 'processed/ml_20condition_retake3_data.csv'), index=False)
Y_df.to_csv(str(OUTPUT_DIR / 'processed/ml_20condition_retake3_label.csv'), index=False)

print("Paired and saved the repeat-measurement dataset.")
print(f"Generated {len(X_df)} samples")


## Construct calibrated-spectrum pairs

Original code cell 6.


In [ ]:
df = pd.read_csv(str(DATA_DIR / 'measurements/s_lambda_correction/calibrated_unknowns.csv'))
col = list(df.columns)
numeric_cols = col[11:11+1873]
wavelength = np.array([float(j) for j in numeric_cols])


ranges = [
    {"x0": 405.24, "x1": 407.245},
    {"x0": 367.942, "x1": 369.659}
]
col_map = {round(float(c), 3): c for c in numeric_cols}
for r in ranges:
    x0, x1 = r["x0"], r["x1"]
    lower_val = min(wavelength, key=lambda w: abs(w - x0))
    upper_val = min(wavelength, key=lambda w: abs(w - x1))
    lower_col = col_map[round(lower_val, 3)]
    upper_col = col_map[round(upper_val, 3)]
    for w in wavelength:
        if x0 <= w <= x1:
            col_str = col_map.get(round(w, 3))
            if col_str is None or 'Pb' in col_str:
                continue
            y0 = df[lower_col]
            y1 = df[upper_col]
            a = (y1 - y0) / (upper_val - lower_val)
            b = y0 - a * lower_val
            df[col_str] = a * w + b


input_cols = sorted(numeric_cols, key=lambda x: float(x))
target_cols_sorted = sorted(numeric_cols, key=lambda x: float(x))


label_df = pd.read_csv(str(DATA_DIR / 'measurements/20condition_seq1_15/mean.csv'))
label_df = label_df[label_df['Cr'] == 1]


X_data = []
Y_data = []

for _, row in df.iterrows():
    cr, cu, ni, zn = row["Cr"], row["Cu"], row["Ni"], row["Zn"]
    spectrum_x = row[input_cols].values
    X_data.append([cr, cu, ni, zn] + list(spectrum_x))


    spectrum_y = global_y_background.copy()
    for metal in ["Cu", "Ni", "Zn"]:
        sy = metal_peak_cache_y_cr1[metal].get(locals()[metal.lower()])
        if sy is not None:
            for indices in metal_peaks[metal]:
                spectrum_y[indices[0]:indices[1]+1] = sy[indices[0]:indices[1]+1]

    Y_data.append([cr, cu, ni, zn] + list(spectrum_y))


X_array = np.array(X_data)
Y_array = np.array(Y_data)


X_spectra = X_array[:, 4:]
Y_spectra = Y_array[:, 4:]


X_spectra[:, indices_set_to_60000_low] = 60000
Y_spectra[:, indices_set_to_60000_low] = 60000


mask_high_wl = wavelength >= 350
high_indices = np.where(mask_high_wl)[0]
X_spectra[:, high_indices] = np.clip(X_spectra[:, high_indices], None, 60000)
Y_spectra[:, high_indices] = np.clip(Y_spectra[:, high_indices], None, 60000)


mask_condition3 = (wavelength >= 323) & (wavelength <= 325)
indices_condition3 = np.where(mask_condition3)[0]

X_spectra[:, indices_condition3] = 60000
Y_spectra[:, indices_condition3] = 60000


X_array[:, 4:] = X_spectra
Y_array[:, 4:] = Y_spectra


with h5py.File(str(OUTPUT_DIR / 'processed/ml_20condition_correction_data.h5'), "w") as h5f:
    h5f.create_dataset("X_with_meta", data=X_array)
    h5f.create_dataset("X_columns", data=np.array(X_cols, dtype='S'))

with h5py.File(str(OUTPUT_DIR / 'processed/ml_20condition_correction_label.h5'), "w") as h5f:
    h5f.create_dataset("Y_with_meta", data=Y_array)
    h5f.create_dataset("Y_columns", data=np.array(Y_cols, dtype='S'))


X_cols = ["Cr", "Cu", "Ni", "Zn"] + input_cols
Y_cols = ["Cr", "Cu", "Ni", "Zn"] + target_cols_sorted

X_df = pd.DataFrame(X_array, columns=X_cols)
Y_df = pd.DataFrame(Y_array, columns=Y_cols)

X_df.to_csv(str(OUTPUT_DIR / 'processed/ml_20condition_correction_data.csv'), index=False)
Y_df.to_csv(str(OUTPUT_DIR / 'processed/ml_20condition_correction_label.csv'), index=False)

print("Paired and saved the corrected dataset.")
print(f"Generated {len(X_df)} samples")


## Construct wastewater pairs

Original code cell 7.


In [ ]:
df = pd.read_csv(str(DATA_DIR / 'measurements/wastewater2/mean.csv'))
col = list(df.columns)
numeric_cols = col[11:11+1873]
wavelength = np.array([float(j) for j in numeric_cols])


ranges = [
    {"x0": 405.24, "x1": 407.245},
    {"x0": 367.942, "x1": 369.659}
]
col_map = {round(float(c), 3): c for c in numeric_cols}
for r in ranges:
    x0, x1 = r["x0"], r["x1"]
    lower_val = min(wavelength, key=lambda w: abs(w - x0))
    upper_val = min(wavelength, key=lambda w: abs(w - x1))
    lower_col = col_map[round(lower_val, 3)]
    upper_col = col_map[round(upper_val, 3)]
    for w in wavelength:
        if x0 <= w <= x1:
            col_str = col_map.get(round(w, 3))
            if col_str is None or 'Pb' in col_str:
                continue
            y0 = df[lower_col]
            y1 = df[upper_col]
            a = (y1 - y0) / (upper_val - lower_val)
            b = y0 - a * lower_val
            df[col_str] = a * w + b


input_cols = sorted(numeric_cols, key=lambda x: float(x))
target_cols_sorted = sorted(numeric_cols, key=lambda x: float(x))


label_df = pd.read_csv(str(DATA_DIR / 'measurements/20condition_seq1_15/mean.csv'))
label_df = label_df[label_df['Cr'] == 1]


X_data = []
Y_data = []

for _, row in df.iterrows():
    cr, cu, ni, zn = row["Cr"], row["Cu"], row["Ni"], row["Zn"]
    spectrum_x = row[input_cols].values
    X_data.append([cr, cu, ni, zn] + list(spectrum_x))


    spectrum_y = global_y_background.copy()
    for metal in ["Cu", "Ni", "Zn"]:
        sy = metal_peak_cache_y_cr1[metal].get(locals()[metal.lower()])
        if sy is not None:
            for indices in metal_peaks[metal]:
                spectrum_y[indices[0]:indices[1]+1] = sy[indices[0]:indices[1]+1]

    Y_data.append([cr, cu, ni, zn] + list(spectrum_y))


X_array = np.array(X_data)
Y_array = np.array(Y_data)


X_spectra = X_array[:, 4:]
Y_spectra = Y_array[:, 4:]


X_spectra[:, indices_set_to_60000_low] = 60000
Y_spectra[:, indices_set_to_60000_low] = 60000


mask_high_wl = wavelength >= 350
high_indices = np.where(mask_high_wl)[0]
X_spectra[:, high_indices] = np.clip(X_spectra[:, high_indices], None, 60000)
Y_spectra[:, high_indices] = np.clip(Y_spectra[:, high_indices], None, 60000)


mask_condition3 = (wavelength >= 323) & (wavelength <= 325)
indices_condition3 = np.where(mask_condition3)[0]

X_spectra[:, indices_condition3] = 60000
Y_spectra[:, indices_condition3] = 60000


X_array[:, 4:] = X_spectra
Y_array[:, 4:] = Y_spectra


with h5py.File(str(OUTPUT_DIR / 'processed/ml_20condition_wastewater2_data.h5'), "w") as h5f:
    h5f.create_dataset("X_with_meta", data=X_array)
    h5f.create_dataset("X_columns", data=np.array(X_cols, dtype='S'))

with h5py.File(str(OUTPUT_DIR / 'processed/ml_20condition_wastewater2_label.h5'), "w") as h5f:
    h5f.create_dataset("Y_with_meta", data=Y_array)
    h5f.create_dataset("Y_columns", data=np.array(Y_cols, dtype='S'))


X_cols = ["Cr", "Cu", "Ni", "Zn"] + input_cols
Y_cols = ["Cr", "Cu", "Ni", "Zn"] + target_cols_sorted

X_df = pd.DataFrame(X_array, columns=X_cols)
Y_df = pd.DataFrame(Y_array, columns=Y_cols)

X_df.to_csv(str(OUTPUT_DIR / 'processed/ml_20condition_wastewater2_data.csv'), index=False)
Y_df.to_csv(str(OUTPUT_DIR / 'processed/ml_20condition_wastewater2_label.csv'), index=False)

print("Paired and saved the wastewater dataset.")
print(f"Generated {len(X_df)} samples")


## Inspect metal-specific wavelength windows

Original code cell 8.


In [ ]:
import matplotlib.pyplot as plt


cr1_group = df[df["Cr"] == 1]
mean_spectrum = cr1_group[input_cols].mean()


wavelengths = np.array([float(w) for w in input_cols])


plt.figure(figsize=(8, 4))
plt.plot(wavelengths, X_array[0, 4:], label="Cr=1 Mean Spectrum", color='black')


def highlight_region(w1, w2, label, color):
    plt.axvspan(w1, w2, color=color, alpha=0.2, label=label)

highlight_region(212.129, 216.777, "Zn", "blue")
highlight_region(229.563, 234.734, "Ni", "green")
highlight_region(326.669, 330.016, "Cu", "red")


plt.xlabel("Wavelength (nm)")
plt.ylabel("Intensity")
plt.title("Mean Spectrum of Cr=1")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()
